In [ ]:
import numpy as np
import pandas as pd
import torch
import os
import time

from transformers import Trainer, TrainingArguments
from datasets import load_dataset
from peft import PromptTuningConfig, get_peft_model
from peft import LoraConfig, get_peft_model, TaskType

from metrics import calculate_meteor_score, calculate_bleu_score
from tools import (inference_batch, get_model, tokenize_function, print_number_of_trainable_model_parameters,
                   format_inference_shots, format_input_texts)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tarefa 1

## <span style="color:#FFA500"> Utilizando novamente o dataset de TESTE e o modelo ajustado na tarefa 2, execute mais uma vez as instruções da coluna “instruction” do dataset. Extraia 2 partes do dataset com 3000 linhas cada uma. Uma das partes será o dataset de treino e a outra, o de teste.Você pode escolher as linhas de cada dataset da forma que preferir.</span>

In [ ]:
dataset_name = 'tatsu-lab/alpaca'

dataset = load_dataset(dataset_name)
dataset['train'].num_rows

In [ ]:
test_size = 3000
train_size = 3000
inference_input_idx = 5

idx = np.random.choice(dataset['train'].shape[0], size=train_size + test_size + inference_input_idx, replace=False)
traind_idx = idx[:train_size]
test_idx = idx[train_size:-inference_input_idx]
inference_input_idx = idx[-inference_input_idx:]
train_dataset = dataset.copy()['train'].select(traind_idx)
test_dataset = dataset.copy()['train'].select(test_idx)
inference_dataset = dataset.copy()['train'].select(inference_input_idx)

print(f'Train data size: {len(train_dataset)}\nTest data size: {len(test_dataset)}\nInference data size: {len(inference_dataset)}')

## <span style="color:#FFA500">Utilizando o modelo indicado, execute as instruções da coluna “instruction” do dataset de TESTE.</span>

### Sem usar template de prompt

In [ ]:
model, tokenizer = get_model(device)
print(print_number_of_trainable_model_parameters(model))

In [ ]:
result_df = pd.DataFrame()
result_df['instruction'] = test_dataset['instruction']
result_df['reference'] = test_dataset['output']
result_df['prediction'] = inference_batch(test_dataset['instruction'], model, tokenizer, device)

In [ ]:
for i in result_df.index[:10]:
    print(f"Instruction: {result_df['instruction'][i]}\nResponse: {result_df['prediction'][i]}\nReference: {result_df['reference'][i]}")
    print('\n' + ('-' * 150))

In [ ]:
general_results = {}
result_df['bleu'] = calculate_bleu_score(result_df['prediction'], result_df['reference'])
result_df['meteor'] = calculate_meteor_score(result_df['prediction'], result_df['reference'])

general_results['without_prompt_and_ft'] = result_df.drop(['instruction', 'prediction', 'reference'],
                                                          axis=1).mean().to_dict()
general_results['without_prompt_and_ft']

In [ ]:
result_df.head()

## <span style="color:#FFA500">Avalie a qualidade do resultado. As respostas estavam corretas? Quais métodos podem ser usados para melhorá-las?</span>

---
<span style="color:#00BFFF">Resposta: Tanto os valores retornados nas métricas de avaliação, quanto os outputs gerados pelo modelo, não são bons... Isso já era um comportamento esperado, pois o modelo "ComCom/gpt2-small" além de pequeno foi treinado apenas para completar texto. Podemos utilizar métodos de prompt engineering, inserir shots de inferência dentro dos inputs para aumentar a chance de o modelo entender o padrão das tarefas solicitadas.</span>

### Usando template de prompt

#### Zero-shot inference

In [ ]:
n_shots = 0
inference_shots = format_inference_shots(inference_dataset, n_shots)
print(inference_shots)

In [ ]:
input_texts_0s = format_input_texts(test_dataset, inference_shots)

result_df = pd.DataFrame()
result_df['instruction'] = test_dataset['instruction']
result_df['reference'] = test_dataset['output']
result_df['prediction'] = inference_batch(input_texts_0s, model, tokenizer, device)

In [ ]:
for i in result_df.index[:10]:
    print(f"Instruction: {result_df['instruction'][i]}\nResponse: {result_df['prediction'][i]}\nReference: {result_df['reference'][i]}")
    print('\n' + ('-' * 150))

In [ ]:
result_df['bleu'] = calculate_bleu_score(result_df['prediction'], result_df['reference'])
result_df['meteor'] = calculate_meteor_score(result_df['prediction'], result_df['reference'])

general_results['prompt_eng_0shot'] = result_df.drop(['instruction', 'prediction', 'reference'],
                                                          axis=1).mean().to_dict()
general_results['prompt_eng_0shot']

In [ ]:
result_df.head()

#### One shot inference

In [ ]:
n_shots = 1
inference_shots = format_inference_shots(inference_dataset, n_shots)
print(inference_shots)

In [ ]:
input_texts_1s = format_input_texts(test_dataset, inference_shots)

result_df = pd.DataFrame()
result_df['instruction'] = test_dataset['instruction']
result_df['reference'] = test_dataset['output']
result_df['prediction'] = inference_batch(input_texts_1s, model, tokenizer, device)

In [ ]:
for i in result_df.index[:10]:
    print(f"Instruction: {result_df['instruction'][i]}\nResponse: {result_df['prediction'][i]}\nReference: {result_df['reference'][i]}")
    print('\n' + ('-' * 150))

In [ ]:
result_df['bleu'] = calculate_bleu_score(result_df['prediction'], result_df['reference'])
result_df['meteor'] = calculate_meteor_score(result_df['prediction'], result_df['reference'])

general_results['prompt_eng_1shot'] = result_df.drop(['instruction', 'prediction', 'reference'],
                                                          axis=1).mean().to_dict()
general_results['prompt_eng_1shot']

In [ ]:
result_df.head()

### Few-shot inference

In [ ]:
n_shots = 3
inference_shots = format_inference_shots(inference_dataset, n_shots)
print(inference_shots)

In [ ]:
input_texts_fs = format_input_texts(test_dataset, inference_shots)

result_df = pd.DataFrame()
result_df['instruction'] = test_dataset['instruction']
result_df['reference'] = test_dataset['output']
result_df['prediction'] = inference_batch(input_texts_fs, model, tokenizer, device)

In [ ]:
for i in result_df.index[:10]:
    print(f"Instruction: {result_df['instruction'][i]}\nResponse: {result_df['prediction'][i]}\nReference: {result_df['reference'][i]}")
    print('\n' + ('-' * 150))

In [ ]:
result_df['bleu'] = calculate_bleu_score(result_df['prediction'], result_df['reference'])
result_df['meteor'] = calculate_meteor_score(result_df['prediction'], result_df['reference'])

general_results['prompt_eng_fs'] = result_df.drop(['instruction', 'prediction', 'reference'],
                                                          axis=1).mean().to_dict()
general_results['prompt_eng_fs']

In [ ]:
result_df.head()

## <span style="color:#FFA500">Avalie a qualidade do resultado. As respostas estavam corretas? Quais métodos podem ser usados para melhorá-las?</span>


---
<span style="color:#00BFFF">Resposta: Ao usar um template de prompt, os resultados se mostraram melhores, mas ainda não é o suficiente.</span>

# Tarefa 2

## <span style="color:#FFA500"> Utilizando o dataset de TREINO, faça o ajuste (fine-tuning) do modelo. Você pode usar técnicas para simplificar o fine-tuning, ajustando um conjunto menor de parâmetros e consumindo menos memória.</span>

## <span style="color:#FFA500"> Utilizando novamente o dataset de TESTE e o modelo ajustado na tarefa 2, execute mais uma vez as instruções da coluna “instruction” do dataset.</span>

# Fine Tuning

In [ ]:
model, tokenizer = get_model(device)

In [ ]:
prompt = """Below is an instruction for a task. First comes ### Instruction, giving the instruction to be followed. Second comes ### Input, giving information that supports the instruction. Write a response that adequately completes the request from ### Response.
### Instruction:
{}

### Input:
{}

### Response:
"""

tokenized_train_dataset = train_dataset.map(lambda example: tokenize_function(example, prompt, tokenizer), batched=False)

### Prompt Tuning with Prompt template

In [ ]:
num_virtual_tokens = 30

peft_config = PromptTuningConfig(
    task_type="CAUSAL_LM",
    num_virtual_tokens=num_virtual_tokens,  # Número de tokens do prompt
    token_dim=model.config.hidden_size,
    num_attention_heads=model.config.num_attention_heads
)

prompt_tuning_model = get_peft_model(model, peft_config)

In [ ]:
print(print_number_of_trainable_model_parameters(prompt_tuning_model))

In [ ]:
output_dir = f'./prompt-tuning-gpt2-alpaca-{str(int(time.time()))}'

peft_training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,
    learning_rate=1e-3,
    logging_steps=100,
    max_steps=100,
    remove_unused_columns=False,
    label_names=["input_ids"]
)
    
peft_trainer = Trainer(
    model=prompt_tuning_model,
    args=peft_training_args,
    train_dataset=tokenized_train_dataset,
)

In [ ]:
peft_trainer.train()
peft_trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir, safe_serialization=True)

In [ ]:
result_df = pd.DataFrame()
result_df['instruction'] = test_dataset['instruction']
result_df['reference'] = test_dataset['output']
result_df['prediction'] = inference_batch(input_texts_0s, prompt_tuning_model, tokenizer, device)

result_df['bleu'] = calculate_bleu_score(result_df['prediction'], result_df['reference'])
result_df['meteor'] = calculate_meteor_score(result_df['prediction'], result_df['reference'])

general_results[f'prompt_tunning_0s'] = result_df.drop(['instruction', 'prediction', 'reference'],
                                                        axis=1).mean().to_dict()
general_results[f'prompt_tunning_0s']

for i in result_df.index[:3]:
    print(f"Instruction: {result_df['instruction'][i]}\nResponse: {result_df['prediction'][i]}\nReference: {result_df['reference'][i]}")
    print('\n' + ('-' * 150))

In [ ]:
result_df.head()

### LoRA fine tuning with Prompt template

In [ ]:
model, tokenizer = get_model(device)

In [ ]:
lora_config = LoraConfig(
    r=512,
    lora_alpha=256,
    target_modules=["c_attn", "c_proj"],
    lora_dropout=0.1,
    bias="lora_only",
    task_type=TaskType.CAUSAL_LM
)

In [ ]:
peft_model = get_peft_model(model, lora_config)

In [ ]:
print(print_number_of_trainable_model_parameters(peft_model))

In [ ]:
output_dir = f'./loraft-gpt2-alpaca-{str(int(time.time()))}'

peft_training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,
    save_safetensors=False,
    learning_rate=1e-3, # Taxa deve ser maior que no full fine tunning
    logging_steps=100,
    max_steps=100   
)
    
peft_trainer = Trainer(
    model=peft_model,
    args=peft_training_args,
    train_dataset=tokenized_train_dataset,
)


In [ ]:
peft_trainer.train()
peft_trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir, safe_serialization=True)

In [ ]:
result_df = pd.DataFrame()
result_df['instruction'] = test_dataset['instruction']
result_df['reference'] = test_dataset['output']
result_df['prediction'] = inference_batch(input_texts_0s, peft_model, tokenizer, device)

result_df['bleu'] = calculate_bleu_score(result_df['prediction'], result_df['reference'])
result_df['meteor'] = calculate_meteor_score(result_df['prediction'], result_df['reference'])

general_results[f'prompt_eng_and_lora_0s'] = result_df.drop(['instruction', 'prediction', 'reference'],
                                                        axis=1).mean().to_dict()
general_results[f'prompt_eng_and_lora_0s']

for i in result_df.index[:3]:
    print(f"Instruction: {result_df['instruction'][i]}\nResponse: {result_df['prediction'][i]}\nReference: {result_df['reference'][i]}")
    print('\n' + ('-' * 150))

### Full fine tuning with Prompt template

In [ ]:
model, tokenizer = get_model(device)

In [ ]:
print(print_number_of_trainable_model_parameters(model))

In [ ]:
output_dir = f'./fullft-gpt2-alpaca-{str(int(time.time()))}'

peft_training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,
    save_safetensors=False,
    learning_rate=1e-4,
    logging_steps=100,
    max_steps=100
)
    
trainer = Trainer(
    model=model,
    args=peft_training_args,
    train_dataset=tokenized_train_dataset,
)


In [ ]:
trainer.train()
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir, safe_serialization=True)

In [ ]:
result_df = pd.DataFrame()
result_df['instruction'] = test_dataset['instruction']
result_df['reference'] = test_dataset['output']
result_df['prediction'] = inference_batch(input_texts_0s, model, tokenizer, device)

result_df['bleu'] = calculate_bleu_score(result_df['prediction'], result_df['reference'])
result_df['meteor'] = calculate_meteor_score(result_df['prediction'], result_df['reference'])

general_results[f'prompt_eng_and_full_ft_0s'] = result_df.drop(['instruction', 'prediction', 'reference'],
                                                        axis=1).mean().to_dict()
general_results[f'prompt_eng_and_full_ft_0s']

for i in result_df.index[:3]:
    print(f"Instruction: {result_df['instruction'][i]}\nResponse: {result_df['prediction'][i]}\nReference: {result_df['reference'][i]}")
    print('\n' + ('-' * 150))

## <span style="color:#FFA500">Interprete os resultados em comparação com os obtidos na tarefa 1.</span>

In [ ]:
general_results_df = pd.DataFrame(general_results).T
general_results_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 10))
for i, metric in enumerate(general_results_df.columns):
    plt.subplot(1,2,i+1)
    # sns.barplot(general_results_df, x=metric)
    general_results_df.T[metric].plot(kind='bar')

<span style="color:#00BFFF">Resposta: Mesmo após os processos de fine tuning testados, os resultados não melhoraram o suficiente para que o modelo se adaptasse a seguir as instruções. Além disso, o conjunto de dados "tatsu-lab/alpaca" traz diversos tipos de tarefas diferentes (não categorizadas), dificultando ainda mais o fine tuning do modelo para tarefas específicas. Seria necessário termos muitos registros para cada tipo específico de tarefa, para que o modelo ficasse bom em todas elas. Uma alternativa de melhoria seria aumentar o conjunto de dados utilizados para treinamento, outra seria utilizar few-shot inferences com inferências específicas para a tarefa que deve ser solucionada.</span>

## <span style="color:#FFA500">Se você precisasse agrupar as perguntas que tratam de assuntos semelhantes nos 2 datasets, como faria?</span>
---
<span style="color:#00BFFF">Resposta: Poderia gerar embeddings semânticos das entradas da coluna "instruction" usando um modelo de embeddings semântico, que transforma cada frase em um vetor numérico representando seu significado. Com esses vetores, é possível aplicar um algoritmo de clusterização (como KMeans, DBSCAN ou HDBSCAN) para identificar grupos de instruções com temas ou propósitos parecidos, como traduções, resumos ou geração de código. Isso permite organizar o dataset por tipo de tarefa, facilitando análises, seleção de exemplos e melhorias no modelo.</span>